## Data Cleaning 

In this task I will clean the country column and parse the date column in the **store_income_data_task.csv** file.

In [17]:
# Import libraries
import pandas as pd
import numpy as np
import fuzzywuzzy
from fuzzywuzzy import process
import datetime

In [2]:
# Load up store_income_data_task.csv.
# Using a try-except block to catch and handle potential errors.
try:
    # Read in the 'store_income_data_task.csv' file.
   income_df = pd.read_csv('store_income_data_task.csv')
# Using 'except FileNotFoundError' to handle the case where the
# 'store_income_data_task.csv' file does not exist and display an
# informative message.
except FileNotFoundError:
   print("Error: store_income_data_task.csv not found.")
    # Exit the program if the file is not found.
   exit()

1. Take a look at all the unique values in the "country" column. Then, convert the column to lowercase and remove any trailing white spaces.

In [3]:
# Take a look at all the unique values in the 'country' column.
countries = income_df['country'].unique()
print(f"There are {len(countries)} unique countries.")
countries

There are 77 unique countries.


array(['United States/', 'Britain', ' United States', 'Britain/',
       ' United Kingdom', 'U.K.', 'SA ', 'U.K/', 'America',
       'United Kingdom', nan, 'united states', ' S.A.', 'England ', 'UK',
       'S.A./', 'ENGLAND', 'BRITAIN', 'U.K', 'U.K ', 'America/', 'SA.',
       'S.A. ', 'u.k', 'uk', ' ', 'UK.', 'England/', 'england',
       ' Britain', 'united states of america', 'UK/', 'SA/', 'SA',
       'England.', 'UNITED KINGDOM', 'America.', 'S.A..', 's.a.', ' U.K',
       ' United States of America', 'Britain ', 'England', ' SA',
       'United States of America.', 'United States of America/',
       'United States.', 's. africasouth africa', ' England',
       'United Kingdom ', 'United States of America ', ' UK',
       'united kingdom', 'AMERICA', 'America ',
       'UNITED STATES OF AMERICA', ' S. AfricaSouth Africa', 'america',
       'S. AFRICASOUTH AFRICA', 'Britain.', '/', 'United Kingdom.',
       'United States', ' America', 'UNITED STATES', 'sa',
       'United States

In [4]:
# Convert the values in the 'country' column to lowercase.
income_df['country'] = income_df['country'].str.lower()

# Remove any leading and trailing white spaces from the 'country' column.
income_df['country'] = income_df['country'].str.strip()

# Using strip() to remove any leading and trailing './' specified
# characters.
income_df['country'] = income_df['country'].str.strip('./')

# View all the unique values in the 'country' column.
countries = income_df['country'].unique()
print(f"There are {len(countries)} unique countries.")
countries

There are 13 unique countries.


array(['united states', 'britain', 'united kingdom', 'u.k', 'sa',
       'america', nan, 's.a', 'england', 'uk', '',
       'united states of america', 's. africasouth africa'], dtype=object)

2. Note that there should only be three separate countries. Eliminate all variations, so that 'South Africa', 'United Kingdom' and 'United States' are the only three countries.

In [5]:
# Getting the 10 closest matches to "south africa".
matches = fuzzywuzzy.process.extract("south africa", countries, limit=10,
                                     scorer=fuzzywuzzy.fuzz.token_sort_ratio)

# View the matches.
matches

[('s. africasouth africa', 75),
 ('america', 53),
 ('united states of america', 50),
 ('s.a', 40),
 ('britain', 32),
 ('united kingdom', 31),
 ('u.k', 27),
 ('united states', 24),
 ('sa', 14),
 ('uk', 14)]

In [6]:
# Getting the 10 closest matches to "united kingdom".
matches = fuzzywuzzy.process.extract("united kingdom", countries, limit=10,
                                     scorer=fuzzywuzzy.fuzz.token_sort_ratio)

# View the matches.
matches

[('united kingdom', 100),
 ('united states', 52),
 ('united states of america', 47),
 ('u.k', 35),
 ('england', 29),
 (nan, 24),
 ('britain', 19),
 ('s. africasouth africa', 18),
 ('s.a', 12),
 ('uk', 12)]

In [7]:
# Getting the 10 closest matches to "uk".
matches = fuzzywuzzy.process.extract("uk", countries, limit=10,
                                     scorer=fuzzywuzzy.fuzz.token_sort_ratio)

# View the matches.
matches

[('uk', 100),
 ('u.k', 40),
 ('united states', 13),
 ('united kingdom', 12),
 ('s. africasouth africa', 9),
 ('united states of america', 8),
 ('britain', 0),
 ('sa', 0),
 ('america', 0),
 (nan, 0)]

In [8]:
# Getting the top 10 closest matches to "united states of america".
matches = fuzzywuzzy.process.extract("united states of america", countries,
                                     limit=10,
                                     scorer=fuzzywuzzy.fuzz.token_sort_ratio)

# View the matches.
matches

[('united states of america', 100),
 ('united states', 70),
 ('america', 45),
 ('united kingdom', 42),
 ('s. africasouth africa', 41),
 ('britain', 32),
 ('s.a', 22),
 ('england', 19),
 ('u.k', 15),
 (nan, 15)]

In [9]:
# Function that will replace rows in a specified column in the
# DataFrame that match the specified string using fuzzywuzzy above the
# ratio that is provided with the specified string.
def replace_matches(df, column, string_to_match, min_ratio = 90):
    # Getting all the unique strings from the column.
    unique_strings = df[column].unique()

    # Getting the top 10 closest matches to the input string.
    matches = fuzzywuzzy.process.extract(string_to_match, unique_strings, limit=10,
                                         scorer=fuzzywuzzy.fuzz.token_sort_ratio)

    # Only get matches with a ratio above or equal to the minimum ratio.
    close_matches = [match[0] for match in matches if match[1] >= min_ratio]

    # Using .isin() to get the rows of all the close_matches in the
    # Dataframe.
    rows_with_matches = df[column].isin(close_matches)

    # Replace all rows that have close matches with the input matches.
    df.loc[rows_with_matches, column] = string_to_match

    # Display a success message when the function is done.
    print(f"Replacement for '{string_to_match}' complete!")

In [10]:
replace_matches(df=income_df, column='country', string_to_match="south africa",
                min_ratio=75)

replace_matches(df=income_df, column='country',
                string_to_match="united states of america", min_ratio=45)

replace_matches(df=income_df, column='country', string_to_match="uk",
                min_ratio=40)

Replacement for 'south africa' complete!
Replacement for 'united states of america' complete!
Replacement for 'uk' complete!


In [11]:
# Take a look at all the unique values in the 'country' column.
countries = income_df['country'].unique()
print(f"There are {len(countries)} unique countries.")
countries

There are 10 unique countries.


array(['united states of america', 'britain', 'united kingdom', 'uk',
       'sa', nan, 's.a', 'england', '', 'south africa'], dtype=object)

In [12]:
income_df.replace('', np.nan, inplace=True)

# Using dropna() to remove any rows with missing values.
income_df = income_df.dropna()

# Get all the unique values in the 'country' column.
countries = income_df['country'].unique()
print(f"There are {len(countries)} unique countries.")
countries

There are 8 unique countries.


array(['britain', 'uk', 'sa', 'united kingdom', 's.a',
       'united states of america', 'england', 'south africa'],
      dtype=object)

In [13]:
# Using replace() to replace different input variations of the same
# country with one country name.
income_df.replace(['uk', 'britain', 'england', 'united kingdom'],
                  'United Kingdom', inplace=True)

income_df.replace('united states of america', 'United States', inplace=True)

income_df.replace(['s.a', 'sa', 'south africa'], 'South Africa', inplace=True)


# Get all the unique values in the 'country' column.
countries = income_df['country'].unique()
print(f"There are {len(countries)} unique countries.")
countries

There are 3 unique countries.


array(['United Kingdom', 'South Africa', 'United States'], dtype=object)

3. Create a new column called `days_ago` in the DataFrame that is a copy of the 'date_measured' column but instead it is a number that shows how many days ago it was measured from the current date. Note that the current date can be obtained using `datetime.date.today()`.

In [14]:
# Getting the data type for the 'date_measured' column.
print(income_df['date_measured'].dtype)

object


In [15]:
# Parsing the dates in the 'date_measured' column.
income_df['date_measured'] = pd.to_datetime(income_df['date_measured'],
                                            format='%d-%m-%Y')

# Check the data type of the 'date_measured' column again after parsing
# the dates.
print(income_df['date_measured'].dtype)

datetime64[ns]


In [16]:
# Get the current date using datetime.date.today().
current_date = datetime.date.today()

# Display the current date.
print("The current date is ", current_date)

# Parsing the current date.
current_date = pd.to_datetime(current_date, format='%d-%m-%Y')

# Create a new column called 'days_ago', and calculate the number of
# days ago that each date was measured.
income_df['days_ago'] = (current_date - income_df['date_measured']).dt.days

print(income_df['days_ago'].head())
income_df.head()

The current date is  2026-07-17
3      7375
5      9798
6      9170
9      7311
15    12140
Name: days_ago, dtype: int64


,id,store_name,store_email,department,income,date_measured,country,days_ago
3,4,FIRST REPUBLIC BANK,ecanadine3@fc2.com,Automotive,$8928350.04,2006-05-08,United Kingdom,7375
5,6,"Auburn National Bancorporation, Inc.",ccaldeyroux5@dion.ne.jp,Grocery,$69798987.04,1999-09-19,United Kingdom,9798
6,7,"Interlink Electronics, Inc.",orodenborch6@skyrock.com,Garden,$22521052.79,2001-06-08,South Africa,9170
9,10,"Synopsys, Inc.",lcancellieri9@tmall.com,Electronics,$44091294.62,2006-07-11,United Kingdom,7311
15,16,New Home Company Inc. (The),nhinchcliffef@whitehouse.gov,Shoes,$90808764.99,1993-04-21,United Kingdom,12140
